# Execution Simulation

Compare conservative taker and later passive maker fill assumptions.

## Execution Simulation: Finance-Specific Mechanics

Execution should model two distinct strategy families:

1. **Single-leg fair-value trading**: buy YES or NO when fair value clears the executable ask plus fees/slippage.
2. **Six-leg partition basket arbitrage**: execute all legs in a mutually exclusive temperature partition when the basket bound clears costs.

The partition basket is especially important because it is a structural event-market constraint, not just a predictive signal.

In [ ]:
from pathlib import Path
import polars as pl

basket_path = Path("../artifacts/research/finance_partition_basket_arbitrage.csv")
basket = pl.read_csv(basket_path) if basket_path.exists() else pl.DataFrame()
print("Best short-YES / long-NO basket candidates")
display(basket.sort("short_basket_edge_cents", descending=True).head(10) if basket.height else pl.DataFrame({"status": [f"missing {basket_path}"]}))
print("Best long-YES basket candidates")
display(basket.sort("long_basket_edge_cents", descending=True).head(10) if basket.height else pl.DataFrame({"status": [f"missing {basket_path}"]}))

### Execution Rules To Implement

For a six-leg partition with exactly one outcome paying YES:

- `sum_yes_ask < 100`: buy one YES contract in every leg. Guaranteed payoff is 100 cents, cost is `sum_yes_ask`.
- `sum_yes_bid > 100`: buy one NO contract in every leg. Guaranteed payoff is 500 cents for six legs, cost is `600 - sum_yes_bid`.

Implementation must be all-or-none. If any leg is missing, stale, wider than the configured spread limit, or size-constrained, skip the basket. The simulator should support partial failure analysis, but production strategy should require atomic basket feasibility.